# Process a large table with a flow

## Goal

Some decisions need several steps: check eligibility, choose a policy, calculate a result, and route exceptions. A Rulebricks **flow** connects those steps so that a Spark pipeline can invoke the published decision process for each record.

This notebook processes 20,000 independent requests. Spark distributes the work, while Rulebricks runs the flow. The result is a saved table containing every source record and its outcome.

| Source | Processing | Saved result |
| --- | --- | --- |
| 20,000 requests | Small groups of requests sent to one flow version | 20,000 aligned outcomes, including individual failures |

This is **stateless batch processing**: each record can be evaluated independently. No Rulebricks state store is needed.

## Setup

Use the [installation guide](../docs/getting-started.md). Edit the API URL, flow identifier, version, and output location below. Use shared storage on a cluster; the local path is for local Spark. The API key comes from `RULEBRICKS_API_KEY` or your secret manager.

Use an isolated flow with this example behavior:

| Inputs | Successful output | Example failure |
| --- | --- | --- |
| `request_id`: string, `amount`: number, `country`: string | `decision="approve"` for amounts below 100 in US/CA, otherwise `"review"`; `score=amount` | A negative amount returns `{"error": "Amount must be non-negative"}` for that record |

The failure stays in its own position in the batch response. Your real flow can have different branches and output fields; its result is kept as JSON.

In [ ]:
import os
import uuid

from pyspark.sql import SparkSession, functions

from rulebricks_spark import apply_flow

spark = SparkSession.getActiveSession() or SparkSession.builder.appName("bulk-flows").getOrCreate()
api_key = os.environ["RULEBRICKS_API_KEY"]
base_url = "https://rulebricks.acme.com/api/v1"
flow_slug = "order-routing-flow"
flow_version = "1"
output_root = "file:///tmp/rulebricks-examples"
run_id = str(uuid.uuid4())
result_path = f"{output_root}/{run_id}/decisions"
print("Results:", result_path)

## 1. Choose a batch of records

Spark generates the example table without collecting it on the notebook machine. One record has a negative amount so we can see a failed calculation alongside successful decisions.

| Column | Purpose |
| --- | --- |
| `request_id` | Match each decision to its source record |
| `amount`, `country` | Inputs used by the flow |

Replace this cell with a fixed table snapshot or batch read in your pipeline.

In [ ]:
record_count = 20_000
source = spark.range(record_count).select(
    functions.concat(functions.lit(f"{run_id}-"), functions.col("id").cast("string")).alias(
        "request_id"
    ),
    functions.when(functions.col("id") == 137, -1)
    .otherwise(functions.col("id") % 200)
    .cast("double")
    .alias("amount"),
    functions.when(functions.col("id") % 2 == 0, "US").otherwise("CA").alias("country"),
)

## 2. Choose how much work to send at once

A **chunk** is one HTTP request containing several records. Each Spark task sends one chunk and waits for its response before sending another.

| Setting | Example value | Why change it? |
| --- | --- | --- |
| `execution_partitions` | `4` | Allow more independent requests to run at once, within service capacity |
| `max_batch_bytes` | 5 MiB | Use larger chunks to reduce request overhead when the service accepts them |
| `max_batch_rows` | `1000` | Limit work and response size when individual records are expensive |
| `timeout_seconds` | `300` | Give a synchronous chunk enough time to finish |

The byte or row limit reached first closes the chunk. Four partitions normally allow up to four requests in flight; available Spark slots may reduce that. Task replay, overlapping jobs, or server work continuing after a timeout can exceed this nominal limit.

Start with a representative sample and increase concurrency while watching latency and service errors.

## 3. Execute and save once

`apply_flow()` adds two columns; it leaves the source columns intact.

| Column added | Meaning |
| --- | --- |
| `rulebricks_result` | The flow's JSON response, which may have different fields for different records |
| `rulebricks_meta` | Outcome, error message, version, timing, and available execution IDs |

Writing starts the execution. The default transient retries suit this stateless example. Flows that change shared data need the [stateful processing](05-stateful-processing.ipynb) pattern.

In [ ]:
evaluated = apply_flow(
    source,
    flow_slug,
    version=flow_version,
    api_key=api_key,
    base_url=base_url,
    correlation_id_path="request_id",
    execution_partitions=4,
    max_batch_bytes=5 * 1024 * 1024,
    max_batch_rows=1000,
    timeout_seconds=300,
)
evaluated.write.format("delta").mode("errorifexists").save(result_path)
saved = spark.read.format("delta").load(result_path)

## 4. Check the saved outcomes

All following actions read the saved table. They do not repeat the flow calls.

| Outcome | Result | Where to investigate |
| --- | --- | --- |
| Successful decision | JSON | Your downstream decision columns |
| Calculation failed for one record | Null | That row's error message and execution ID |
| Request failed | Null for affected records | Request error details in metadata |

The example should retain 19,999 successful records and one calculation error. A failed item does not discard its neighbors.

In [ ]:
assert saved.count() == record_count
saved.groupBy("rulebricks_meta.status", "rulebricks_meta.error.type").count().show(truncate=False)
failed = saved.filter("rulebricks_meta.status = 'error'")
assert failed.count() == 1
failed.select(
    "request_id",
    "amount",
    "rulebricks_meta.error",
    "rulebricks_meta.execution_id",
    "rulebricks_meta.item_execution_id",
    "rulebricks_meta.trace_id",
).show(20, truncate=False)

saved.filter("rulebricks_meta.status = 'ok'").select(
    "request_id",
    functions.get_json_object("rulebricks_result", "$.data.decision").alias("decision"),
).show(10, truncate=False)

## Next steps

For your own flow, begin with the saved JSON and select the fields your downstream table needs. The [results guide](../docs/working-with-results.md) shows how to select and parse saved JSON with ordinary Spark expressions.

Keep the run path with your pipeline run record. Repeated actions on `evaluated` can execute the flow again; caching alone is not a durable execution boundary. See [reliability and audit](../docs/reliability-and-audit.md) when preparing a production run.